In [1]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user="hr",
        password="basededatos",
        dsn="localhost:1521/XE"  
    )
    cursor = conn.cursor()
    cursor.callproc("dbms_output.enable")


except oracledb.Error as e:
    print(f"Error detectado: {e}")

In [ ]:
try:
    cursor.execute(
    f"""
    
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

crear un procedimiento que llame a una funcion f_localidad y muestre el nombre del empleado y el valor de f_localidad

f_localidad dado un id del empleado da la localidad del empleado

In [8]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION f_localidad
    (p_emp_id IN employees.employee_id%TYPE)
    RETURN VARCHAR2
    IS
        v_localidad locations.city%TYPE;
    BEGIN
        SELECT NVL(l.city, 'No tiene localidad')
        INTO v_localidad
        FROM employees e
        LEFT JOIN departments d ON e.department_id = d.department_id
        LEFT JOIN locations l ON d.location_id = l.location_id
        WHERE e.employee_id = p_emp_id;
        RETURN v_localidad;
    END f_localidad;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE pro_llamar_localidad
    IS
        CURSOR empleados IS
            SELECT first_name || ' ' || last_name AS nombre, f_localidad(employee_id) AS localidad
            FROM employees;
    BEGIN
        FOR empleado IN empleados LOOP
            DBMS_OUTPUT.PUT_LINE('Nombre: ' || empleado.nombre || ', Localidad: ' || empleado.localidad);
        END LOOP;
    END pro_llamar_localidad;
    """
    )

    cursor.execute(
    """
    BEGIN
        pro_llamar_localidad;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Nombre: Steven King, Localidad: Seattle
Nombre: Neena Kochhar, Localidad: Seattle
Nombre: Lex De Haan, Localidad: Seattle
Nombre: Alexander Hunold, Localidad: Southlake
Nombre: Bruce Ernst, Localidad: Southlake
Nombre: David Austin, Localidad: Southlake
Nombre: Valli Pataballa, Localidad: Southlake
Nombre: Diana Lorentz, Localidad: Southlake
Nombre: Nancy Greenberg, Localidad: Seattle
Nombre: Daniel Faviet, Localidad: Seattle
Nombre: John Chen, Localidad: Seattle
Nombre: Ismael Sciarra, Localidad: Seattle
Nombre: Jose Manuel Urman, Localidad: Seattle
Nombre: Luis Popp, Localidad: Seattle
Nombre: Den Raphaely, Localidad: Seattle
Nombre: Alexander Khoo, Localidad: Seattle
Nombre: Shelli Baida, Localidad: Seattle
Nombre: Sigal Tobias, Localidad: Seattle
Nombre: Guy Himuro, Localidad: Seattle
Nombre: Karen Colmenares, Localidad: Seattle
Nombre: Matthew Weiss, Localidad: South San Francisco
Nombre: Adam Fripp, Localidad: South San Francisco
Nombre: Payam Kaufling, Localidad: South San Franc

Instrucciones específicas:

Modularización de Cálculos (Funciones):

Cree una función llamada salario_dep que reciba el ID de un departamento y retorne la suma total de salarios de todos los empleados que pertenecen a él. Si el departamento no tiene empleados, debe retornar 0.

Cree una función llamada compa_dep que reciba el ID de un departamento y calcule cuántos compañeros tiene un empleado en ese departamento (es decir, el conteo total de empleados del departamento menos uno). Asegúrese de que el resultado nunca sea negativo.

Procesamiento de Datos (Procedimiento):

Cree un procedimiento almacenado llamado proceso.

Dentro del procedimiento, declare un Cursor Explícito que obtenga el ID del empleado, el nombre del departamento y el ID del departamento, realizando un JOIN entre las tablas employees y departments.

Utilice un ciclo de control LOOP manual (con OPEN, FETCH y CLOSE) para recorrer el cursor. No se permite el uso de FOR LOOP automático para esta sección.

Para cada fila recuperada, imprima en consola (vía DBMS_OUTPUT) una línea con el siguiente formato: Emp: [ID] | Dept: [Nombre] | Compañeros: [Resultado Función 2] | Total Dept: [Resultado Función 1]

In [12]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION salario_dep
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_salario NUMBER;
    BEGIN
        SELECT NVL(SUM(salary),0)
        INTO v_salario
        FROM employees
        WHERE department_id = p_dep_id;
        RETURN v_salario;
    END salario_dep;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION compa_dep
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_compas NUMBER;
    BEGIN
        SELECT COUNT(*)-1
        INTO v_compas
        FROM employees
        WHERE department_id = p_dep_id;
        RETURN CASE
            WHEN v_compas > 0 THEN v_compas
            ELSE 0
            END;
    END compa_dep;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE pro_salarios_compas
    IS
        CURSOR empleados IS
            SELECT e.first_name || ' ' || e.last_name AS nombre, d.department_name AS departamento, 
                compa_dep(e.department_id) AS compas, salario_dep(e.department_id) AS salarios
            FROM employees e
            JOIN departments d ON e.department_id = d.department_id;
        v_nombre VARCHAR2(100);
        v_departamento departments.department_name%TYPE;
        v_compas NUMBER;
        v_salarios NUMBER;
    BEGIN
        OPEN empleados;
        LOOP
            FETCH empleados INTO v_nombre, v_departamento, v_compas, v_salarios;
            EXIT WHEN empleados%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE('Nombre: ' || v_nombre || ', Departamento: ' || v_departamento || ', Companieros: '
            || TO_CHAR(v_compas) || ', Salarios: ' || TO_CHAR(v_salarios));
        END LOOP; 
        CLOSE empleados;
    END pro_salarios_compas;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        pro_salarios_compas;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Nombre: Jennifer Whalen, Departamento: Administration, Companieros: 0, Salarios: 4400
Nombre: Pat Fay, Departamento: Marketing, Companieros: 1, Salarios: 19000
Nombre: Michael Hartstein, Departamento: Marketing, Companieros: 1, Salarios: 19000
Nombre: Alexander Khoo, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Den Raphaely, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Shelli Baida, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Sigal Tobias, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Karen Colmenares, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Guy Himuro, Departamento: Purchasing, Companieros: 5, Salarios: 24900
Nombre: Susan Mavris, Departamento: Human Resources, Companieros: 0, Salarios: 6500
Nombre: Sarah Bell, Departamento: Shipping, Companieros: 6, Salarios: 41200
Nombre: Britney Everett, Departamento: Shipping, Companieros: 6, Salarios: 41200
Nombre: Shanta Vollman, Departa

listar los empleados, numero de companieros, y comprobar por cada empleado si el salario esta en los rangos

In [17]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION mensaje
    (p_salario IN employees.salary%TYPE) 
    RETURN VARCHAR2
    IS
    BEGIN
        RETURN CASE 
            WHEN p_salario < 1500 THEN 'SUBIR'
            WHEN p_salario BETWEEN 1500 AND 2500 THEN 'SALARIO MEDIO'
            ELSE 'NO SUBIR SALARIO'
        END;
    END mensaje;
    """)

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE pro_mensajes
    IS
        CURSOR empleados IS
            SELECT first_name || ' ' || last_name AS nombre, compa_dep(department_id) AS compas, salary AS salario,
                mensaje(salary) AS mensaje_salario
            FROM employees;
    BEGIN
        FOR empleado IN empleados LOOP
            DBMS_OUTPUT.PUT_LINE('NOMBRE: ' || empleado.nombre || ', COMPAS: ' || TO_CHAR(empleado.compas) || 
            ', SALARIO: ' || TO_CHAR(empleado.salario) || ', MENSAJE: ' || empleado.mensaje_salario);
        END LOOP;
    END pro_mensajes;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        pro_mensajes;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

NOMBRE: Steven King, COMPAS: 2, SALARIO: 32044, MENSAJE: NO SUBIR SALARIO
NOMBRE: Neena Kochhar, COMPAS: 2, SALARIO: 17200, MENSAJE: NO SUBIR SALARIO
NOMBRE: Lex De Haan, COMPAS: 2, SALARIO: 17000, MENSAJE: NO SUBIR SALARIO
NOMBRE: Alexander Hunold, COMPAS: 4, SALARIO: 9000, MENSAJE: NO SUBIR SALARIO
NOMBRE: Bruce Ernst, COMPAS: 4, SALARIO: 6000, MENSAJE: NO SUBIR SALARIO
NOMBRE: David Austin, COMPAS: 4, SALARIO: 4800, MENSAJE: NO SUBIR SALARIO
NOMBRE: Valli Pataballa, COMPAS: 4, SALARIO: 4800, MENSAJE: NO SUBIR SALARIO
NOMBRE: Diana Lorentz, COMPAS: 4, SALARIO: 4200, MENSAJE: NO SUBIR SALARIO
NOMBRE: Nancy Greenberg, COMPAS: 5, SALARIO: 12000, MENSAJE: NO SUBIR SALARIO
NOMBRE: Daniel Faviet, COMPAS: 5, SALARIO: 9000, MENSAJE: NO SUBIR SALARIO
NOMBRE: John Chen, COMPAS: 5, SALARIO: 8200, MENSAJE: NO SUBIR SALARIO
NOMBRE: Ismael Sciarra, COMPAS: 5, SALARIO: 7700, MENSAJE: NO SUBIR SALARIO
NOMBRE: Jose Manuel Urman, COMPAS: 5, SALARIO: 7800, MENSAJE: NO SUBIR SALARIO
NOMBRE: Luis Popp, C

dado un codigo de empleado cambiar su salario en 100
guardar en una tabla de auditoria el usuario, hora y empleado

In [36]:
cursor.execute(
    f"""
    DROP TABLE auditoria
    """
    )

cursor.execute(
    f"""
    CREATE TABLE auditoria (
        usuario VARCHAR(30),
        fecha   DATE,
        empleado    VARCHAR(30)
    )
    """
    )

In [39]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE TRIGGER tr_actualizar
    BEFORE UPDATE ON employees
    FOR EACH ROW
    BEGIN
        INSERT INTO auditoria
        (usuario, fecha, empleado)
        VALUES
        (USER, SYSDATE, :new.employee_id);
    END tr_actualizar;
    """
    )

    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE pro_actualizar
    (p_emp_id IN VARCHAR2) -- Recibimos como VARCHAR2 para poder validar internamente
IS
    e_no_actualizo EXCEPTION;
    v_id_num       NUMBER; -- Variable auxiliar para la conversión
BEGIN
    -- 1. Intentamos convertir el parámetro a número
    -- Si p_emp_id es 'xd', aquí saltará al bloque EXCEPTION (WHEN OTHERS)
    v_id_num := TO_NUMBER(p_emp_id);

    -- 2. Realizamos el UPDATE con el valor ya convertido
    UPDATE employees
    SET salary = salary + 100
    WHERE employee_id = v_id_num;

    -- 3. Validamos si el ID existía en la tabla
    IF SQL%ROWCOUNT = 0 THEN
        RAISE e_no_actualizo;
    END IF;

    DBMS_OUTPUT.PUT_LINE('Actualización exitosa para el ID: ' || v_id_num);

EXCEPTION
    WHEN e_no_actualizo THEN
        DBMS_OUTPUT.PUT_LINE('ERROR: El usuario ' || p_emp_id || ' no existe en la tabla.');
    
    WHEN OTHERS THEN
        -- Aquí capturamos el ORA-06502 (Error de valor en PL/SQL) o ORA-01722 (SQL)
        -- Usamos ABS para comparar el valor absoluto si prefieres, o simplemente los números
        IF SQLCODE = -6502 OR SQLCODE = -1722 THEN
            DBMS_OUTPUT.PUT_LINE('ERROR: El ID "' || p_emp_id || '" no tiene un formato numérico válido.');
        ELSE
            -- Por si sale un error de Trigger u otro problema
            DBMS_OUTPUT.PUT_LINE('ERROR INESPERADO: ' || SQLERRM);
        END IF;
END pro_actualizar;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        pro_actualizar(100);
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Actualización exitosa para el ID: 100
